In [2]:
!wandb login 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


wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


# DATA PIPLEINES 

# 1- Data Ingestion

In [3]:
import pandas as pd
import wandb

# URL for a sample retail sales dataset hosted by the UCI ML repository
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

def ingest_raw_data():
    """
    Step 1 — Data Ingestion Pipeline

    This function:
    1. Starts a W&B run for tracking the ingestion job
    2. Downloads the raw dataset from a remote URL
    3. Saves a local copy (CSV) for reproducibility
    4. Logs the file as a W&B Artifact for versioned data tracking
    """

    # Start a W&B run
    # project → logical workspace name in W&B
    # job_type → helps identify this run as a data ingestion step in the pipeline
    with wandb.init(project="sales-forecasting", job_type="ingest-data") as run:

        # -------------------------------
        # Download the raw dataset
        # -------------------------------
        # pandas reads the Excel file directly from the URL into a DataFrame
        print("Downloading raw data...")
        df = pd.read_excel(DATA_URL)

        # -------------------------------
        # Save a local copy of the dataset
        # -------------------------------
        # This ensures:
        # - reproducibility
        # - faster downstream access
        # - a fixed snapshot for artifact versioning
        filename = "raw_sales_data.csv"
        df.to_csv(filename, index=False)

        # -------------------------------
        # Create a W&B Artifact
        # -------------------------------
        # Artifacts are versioned objects (datasets, models, etc.)
        # name → unique identifier inside the project
        # type → logical grouping (raw_dataset, processed_dataset, model, etc.)
        # description → human-readable metadata for lineage tracking
        raw_data_artifact = wandb.Artifact(
            name="raw-sales-data",
            type="raw_dataset",
            description="Raw online retail sales data from UCI."
        )

        # Attach the local file to the artifact
        raw_data_artifact.add_file(filename)

        # Log the artifact to W&B
        # This uploads the file and creates a versioned dataset entry
        run.log_artifact(raw_data_artifact)

        print("Raw data artifact logged.")

# Execute the ingestion step
ingest_raw_data()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: silvapi1994 (silvapi1994-karabuk-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Raw data artifact logged.


In [4]:
df = pd.read_csv("raw_sales_data.csv")  
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
len(df)

541909

In [7]:
missing_customer_rows = df["CustomerID"].isna().sum()
missing_customer_rows

np.int64(135080)

In [8]:
negative_quality_rows = (df["Quantity"]<=0).sum()
negative_quality_rows

np.int64(10624)

# 2- Data Processing

In [9]:
import pandas as pd
import wandb

def preprocess_data():
    # Start a W&B run to track this preprocessing step in the pipeline
    # job_type helps identify this as the feature engineering / preprocessing stage
    with wandb.init(project="sales-forecasting",
                    job_type="preprocess-data") as run:

        # ---------------------------------------------
        # Load the versioned RAW dataset from W&B
        # ---------------------------------------------
        # This guarantees reproducibility (no local random files)
        raw_artifact = run.use_artifact('raw-sales-data:latest')
        raw_data_dir = raw_artifact.download()

        # Read the raw CSV into a pandas DataFrame
        df = pd.read_csv(f"{raw_data_dir}/raw_sales_data.csv")

        # Store original row count for data quality monitoring
        original_row_count = len(df)

        # ---------------------------------------------
        # DATA CLEANING (remove invalid retail records)
        # ---------------------------------------------

        # Count rows where CustomerID is missing (often anonymous or corrupted records)
        missing_customer_rows = df["CustomerID"].isna().sum()

        # Drop rows with missing CustomerID
        df = df.dropna(subset=["CustomerID"])

        # Count rows with zero or negative quantity (returns or data errors)
        negative_quantity_rows = (df["Quantity"] <= 0).sum()

        # Keep only positive quantities (actual purchases)
        df = df[df["Quantity"] > 0]

        # Count rows with zero or negative price (invalid transactions)
        nonpositive_price_rows = (df["UnitPrice"] <= 0).sum()

        # Keep only rows with positive unit price
        df = df[df["UnitPrice"] > 0]

        # Row count after cleaning
        cleaned_row_count = len(df)

        # ---------------------------------------------
        # TIME SERIES AGGREGATION (core transformation)
        # ---------------------------------------------

        # Convert InvoiceDate column to pandas datetime for time-based operations
        df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

        # Compute revenue per transaction line
        # Sales = Quantity * UnitPrice
        df["Sales"] = df["Quantity"] * df["UnitPrice"]

        # --------------------------------------------------
        # Convert transactional data → DAILY time series
        # Steps:
        # 1. Set InvoiceDate as time index
        # 2. Select the Sales column
        # 3. Resample by day ("D")
        # 4. Sum all transactions that occur on the same day
        # --------------------------------------------------
        daily_sales = df.set_index("InvoiceDate")["Sales"] \
                        .resample("D").sum().reset_index()

        # Rename columns for clarity (model-friendly naming)
        daily_sales = daily_sales.rename(columns={
            "InvoiceDate": "Date",
            "Sales": "TotalSales"
        })

        # --------------------------------------------------
        # Ensure continuous daily timeline
        # asfreq("D") creates missing calendar days
        # fill_value=0 assigns zero sales to days with no transactions
        # This is REQUIRED for most forecasting models
        # --------------------------------------------------
        daily_sales = daily_sales.set_index("Date") \
                                 .asfreq("D", fill_value=0) \
                                 .reset_index()

        # ---------------------------------------------
        # TIME SERIES METADATA (useful for monitoring)
        # ---------------------------------------------
        total_days = len(daily_sales)  # total number of daily time steps
        non_zero_days = (daily_sales["TotalSales"] > 0).sum()  # days with actual sales
        date_start = daily_sales["Date"].min()  # first date in the series
        date_end = daily_sales["Date"].max()    # last date in the series

        # ---------------------------------------------
        # TRAIN / VALIDATION SPLIT (chronological)
        # ---------------------------------------------
        # Use the last 60 days as validation to simulate future forecasting
        # No random split for time series
        train_df = daily_sales[:-60]
        val_df = daily_sales[-60:]

        # Save processed datasets locally
        train_df.to_csv("train.csv", index=False)
        val_df.to_csv("validation.csv", index=False)

        # ---------------------------------------------
        # LOG DATA QUALITY METRICS TO W&B
        # Helps monitor upstream data drift and pipeline health
        # ---------------------------------------------
        wandb.log({
            "original_row_count": original_row_count,
            "cleaned_row_count": cleaned_row_count,
            "rows_dropped_missing_customer": missing_customer_rows,
            "rows_dropped_negative_quantity": negative_quantity_rows,
            "rows_dropped_nonpositive_price": nonpositive_price_rows,
            "total_days": total_days,
            "non_zero_days": non_zero_days,
            "date_start": date_start,
            "date_end": date_end,
        })

        # ---------------------------------------------
        # LOG PROCESSED DATASET AS A NEW W&B ARTIFACT
        # This creates lineage:
        # raw-sales-data → processed-sales-data
        # ---------------------------------------------
        processed_data_artifact = wandb.Artifact(
            "processed-sales-data",
            type="processed_dataset",
            description="Daily aggregated sales data, split into train/validation."
        )

        # Attach processed files
        processed_data_artifact.add_file("train.csv")
        processed_data_artifact.add_file("validation.csv")

        # Upload artifact to W&B
        run.log_artifact(processed_data_artifact)

        print("Processed data artifact logged.")

# Execute preprocessing pipeline step
preprocess_data()


wandb:   1 of 1 files downloaded.  


Processed data artifact logged.


cleaned_row_count,▁
non_zero_days,▁
original_row_count,▁
rows_dropped_missing_customer,▁
rows_dropped_negative_quantity,▁
rows_dropped_nonpositive_price,▁
total_days,▁
cleaned_row_count,397884
date_end,2011-12-09T00:00:00
date_start,2010-12-01T00:00:00
non_zero_days,305


# PyTorch Intrgration

In [11]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import wandb
import random
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# DATASET: Converts a univariate time series into
# (sequence → next value) training samples for LSTM
# ---------------------------------------------------------
class SalesDataset(torch.utils.data.Dataset):
    def __init__(self, data, sequence_length):
        # Convert input array to a 1D float tensor
        # shape: [N]
        self.data = torch.FloatTensor(data).view(-1)

        # Number of past time steps used as input
        # e.g. sequence_length = 30 → use last 30 days to predict next day
        self.sequence_length = sequence_length

    def __len__(self):
        # Total number of training samples that can be generated
        # For each sample we need:
        # sequence_length inputs + 1 target
        return len(self.data) - self.sequence_length - 1

    def __getitem__(self, index):
        # Input sequence: data[index : index + sequence_length]
        # Target value: the next time step after the sequence
        return (
            self.data[index:index + self.sequence_length],          # shape: [seq_len]
            self.data[index + self.sequence_length]                 # scalar target
        )


# ---------------------------------------------------------
# MODEL: Univariate LSTM for next-step forecasting
# ---------------------------------------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_size=1,
                hidden_layer_size=100,
                output_size=1):
                
        super().__init__()

        # Number of hidden units in LSTM
        self.hidden_layer_size = hidden_layer_size

        # LSTM layer
        # input_size = 1 → univariate time series
        # hidden_layer_size = number of LSTM memory cells
        self.lstm = nn.LSTM(input_size, hidden_layer_size)

        # Fully connected layer to map LSTM output → prediction
        self.linear = nn.Linear(hidden_layer_size, output_size)

        # Initial hidden state (h0) and cell state (c0)
        # Shape: (num_layers, batch_size, hidden_size)
        # Here: single layer, batch size = 1 (stateful training)
        self.hidden_cell = (
            torch.zeros(1, 1, self.hidden_layer_size),
            torch.zeros(1, 1, self.hidden_layer_size)
        )

    def forward(self, input_seq):
        # input_seq arrives as:
        # shape: [batch_size, seq_len, 1]
        # LSTM expects: [seq_len, batch_size, input_size]
        # so we permute dimensions
        lstm_out, self.hidden_cell = self.lstm(
            input_seq.permute(1, 0, 2),  # → [seq_len, batch, input_size]
            self.hidden_cell
        )

        # Take the output of the last time step
        # lstm_out shape: [seq_len, batch_size, hidden_size]
        # lstm_out[-1] → last time step → [batch_size, hidden_size]
        predictions = self.linear(lstm_out[-1])

        # Output shape: [batch_size, 1]
        return predictions


# ---------------------------------------------------------
# REPRODUCIBILITY: Fix random seeds across libraries
# ---------------------------------------------------------
def set_seed(seed):
    # Python built-in random
    random.seed(seed)

    # Ensures deterministic hashing (important for reproducibility)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # NumPy random seed
    np.random.seed(seed)

    # PyTorch CPU random seed
    torch.manual_seed(seed)

    # (Optional for full determinism on GPU)
    # torch.cuda.manual_seed_all(seed)
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False


# Tracking with W&B

In [12]:
# ---------------------------------------------------------
# MAIN TRAINING PIPELINE: LSTM time-series forecaster
# ---------------------------------------------------------
def train_forecaster():

    # Hyperparameters and reproducibility settings
    config = {
        "learning_rate": 0.0005,
        "epochs": 15,
        "sequence_length": 30,     # use last 30 days to predict next day
        "hidden_layer_size": 50,   # LSTM memory size
        "seed": 1
    }

    # Ensure deterministic behavior across runs
    set_seed(config["seed"])

    # Start a W&B run for the training stage
    with wandb.init(project="sales-forecasting",
                    job_type="training",
                    config=config) as run:

        # Access config through wandb (enables UI tracking)
        config = wandb.config

        # -------------------------------------------------
        # LOAD PROCESSED DATASET (versioned input)
        # processed-sales-data → train.csv / validation.csv
        # -------------------------------------------------
        artifact = run.use_artifact('processed-sales-data:latest')
        artifact_dir = artifact.download()

        train_df = pd.read_csv(f"{artifact_dir}/train.csv")
        val_df = pd.read_csv(f"{artifact_dir}/validation.csv")

        # -------------------------------------------------
        # EXTRACT TARGET SERIES (TotalSales)
        # Ensure numeric and drop corrupted values
        # -------------------------------------------------
        sales_data = pd.to_numeric(train_df["TotalSales"],
                                   errors="coerce").dropna().values

        val_data = pd.to_numeric(val_df["TotalSales"],
                                 errors="coerce").dropna().values

        # -------------------------------------------------
        # NORMALIZATION (fit on train, apply to val)
        # Important for neural networks stability
        # -------------------------------------------------
        scaler = StandardScaler()

        # Fit only on training data to avoid data leakage
        train_scaled = scaler.fit_transform(sales_data.reshape(-1, 1))

        # Apply same transformation to validation
        val_scaled = scaler.transform(val_data.reshape(-1, 1))

        # -------------------------------------------------
        # CREATE SLIDING-WINDOW DATASETS
        # Converts time series → (sequence → next value)
        # -------------------------------------------------
        train_dataset = SalesDataset(train_scaled,
                                     config.sequence_length)

        val_dataset = SalesDataset(val_scaled,
                                   config.sequence_length)

        # DataLoader with batch_size = 1 (stateful LSTM setup)
        train_loader = torch.utils.data.DataLoader(train_dataset,
                                                   batch_size=1,
                                                   shuffle=False)

        val_loader = torch.utils.data.DataLoader(val_dataset,
                                                 batch_size=1,
                                                 shuffle=False)

        # -------------------------------------------------
        # MODEL, LOSS FUNCTION, OPTIMIZER
        # -------------------------------------------------
        model = LSTMModel(hidden_layer_size=config.hidden_layer_size)

        # Mean Squared Error for regression forecasting
        loss_fn = nn.MSELoss()

        # Adam optimizer for faster convergence
        optimizer = torch.optim.Adam(model.parameters(),
                                     lr=config.learning_rate)

        # Track gradients and parameters in W&B
        wandb.watch(model, log_freq=100)

        # -------------------------------------------------
        # TRAINING LOOP
        # -------------------------------------------------
        best_loss = float("inf")

        for epoch in range(config.epochs):

            model.train()
            train_losses = []

            for seq, label in train_loader:

                optimizer.zero_grad()

                # Reset hidden state per sequence (prevents leakage across batches)
                model.hidden_cell = (
                    torch.zeros(1, 1, model.hidden_layer_size),
                    torch.zeros(1, 1, model.hidden_layer_size)
                )

                # Add feature dimension: [batch, seq_len] → [batch, seq_len, 1]
                pred = model(seq.unsqueeze(-1))

                # Ensure target shape matches prediction: [batch, 1]
                loss = loss_fn(pred, label.view(1, 1))

                # Backpropagation
                loss.backward()
                optimizer.step()

                train_losses.append(loss.item())

            # Average training loss for this epoch
            avg_train_loss = np.mean(train_losses)

            # -------------------------------------------------
            # VALIDATION LOOP (no gradient updates)
            # -------------------------------------------------
            model.eval()
            val_losses = []

            with torch.no_grad():
                for seq, label in val_loader:

                    # Reset hidden state for each validation sequence
                    model.hidden_cell = (
                        torch.zeros(1, 1, model.hidden_layer_size),
                        torch.zeros(1, 1, model.hidden_layer_size)
                    )

                    pred = model(seq.unsqueeze(-1))

                    # Note: label is scalar → broadcast handled by MSELoss
                    val_loss = loss_fn(pred, label)

                    val_losses.append(val_loss.item())

            # Average validation loss
            avg_val_loss = np.mean(val_losses)

            # -------------------------------------------------
            # LOG METRICS TO W&B (per epoch)
            # -------------------------------------------------
            wandb.log({
                "epoch": epoch,
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss
            })

            # -------------------------------------------------
            # MODEL CHECKPOINTING (based on best val loss)
            # -------------------------------------------------
            if avg_val_loss < best_loss:
                best_loss = avg_val_loss

                # Bundle model + scaler + config for reproducible inference
                best_model_bundle = {
                    "model_state_dict": model.state_dict(),
                    "scaler": scaler,
                    "config": dict(config)
                }

                # Save locally
                torch.save(best_model_bundle,
                           "best_model_bundle.pth")

                # Log as a W&B MODEL artifact
                artifact = wandb.Artifact(
                    "sales-forecasting-lstm",
                    type="model",
                    metadata={
                        "epoch": epoch,
                        "val_loss": best_loss
                    }
                )

                artifact.add_file("best_model_bundle.pth")

                # Aliases allow easy retrieval: "best", "latest", etc.
                run.log_artifact(artifact,
                                 aliases=["best", f"epoch_{epoch}"])

        print("Training complete!")

# Execute training
train_forecaster()


wandb:   2 of 2 files downloaded.  
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Training complete!


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,██▇▆▄▃▂▂▂▂▂▁▁▁▁
val_loss,█▇▆▂▂▂▂▂▂▂▁▁▁▁▁
epoch,14
train_loss,0.48313
val_loss,0.56556
